# Submission 1 — Project Visualization (5%)

**Course:** RBB2013 Digital Twin — May 2026
**Group project — SmartClean Twin:** Digital Twin of a mobile cleaning robot (topic 2)

**Team Members:**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |
**Repository:** https://github.com/KAI-UTP/smartclean-twin

**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

> The video walks through the whole project: problem and purpose, architecture, live Grafana dashboard, NVIDIA Omniverse 3D twin, the five AI models, what-if simulation, live fault injection, command and control, tests, CI, scaling and persistence.


## 1. Purpose Alignment (rubric: data & visualization related to twin purpose)

**Twin purpose (SMART):** autonomously monitor a cleaning robot, detect unsafe
conditions within 5 seconds, predict maintenance needs, and report cleaning
progress toward 100% room coverage per cycle.

**Measure of success displayed in visualization:** *cleaning coverage %* —
shown as a live gauge and a progress trend on the dashboard, driven by the
twin's state engine. Every panel exists to serve the purpose above — no
decoration panels.


## 2. Grafana Dashboard — 28 panels, 7 control-room sections

URL: `http://localhost:3001/d/smartclean-main` (admin/admin), auto-refresh 5s.

| Section | Panels | Serves the purpose by |
|---|---|---|
| Robot Status Overview | Safety, Mission, AI Health, Anomaly flag, AI Recommendation | one-glance operator awareness |
| Battery & Power | SoC gauge, voltage, discharge rate (derivative) | energy management |
| Motion & Environment | position, obstacle distance | navigation & safety monitoring |
| Motor & Cleaning | current, temperature, coverage gauge + trend, dirt score | equipment condition + **measure of success** |
| AI Predictions & Forecasts | 5 model outputs, RUL gauge + trend, anomaly score, minutes-to-empty/finish | predictive maintenance |
| Statistical Trends | 30s mean/max, 1m mean windows | aggregation view |
| Alarms & Events | active count, rate, full history table | incident review |

Dashboard is provisioned as code (`grafana/dashboards/smartclean_twin.json`)
and baked into the Grafana container — reproducible on any machine.


## 3. NVIDIA Omniverse 3D Twin (second visualization, same data source)

A USD scene updated every 1 s from the same InfluxDB the dashboard reads —
*one source of truth, two visualizations*:

- robot disc moves with live pose; green cone shows heading
- floor tiles turn green as cells are cleaned (3D coverage view)
- battery bar on the robot shrinks and recolours (green→orange→red)
- robot body colour = safety state; EMERGENCY → red + flashing status light
- red obstacle indicator appears in front of the robot during EMERGENCY
- blue breadcrumb trail shows the path taken

Scripts: `omniverse/create_scene.py`, `omniverse/live_update.py`,
`omniverse/fault_demo.py`. Screenshots in `docs/evidence/`.


## 4. Live evidence — the data the visualizations consume

In [1]:
import json, time, urllib.request

INFLUX = "http://localhost:8086/api/v2/query?org=smartclean"
TOKEN = "smartclean-super-secret-token"

def flux_query(q):
    req = urllib.request.Request(INFLUX, data=q.encode(),
        headers={"Authorization": f"Token {TOKEN}",
                 "Content-Type": "application/vnd.flux", "Accept": "application/csv"})
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.read().decode()

def show_last(measurement, range_s=30):
    q = (f'from(bucket: "smartclean_twin") |> range(start: -{range_s}s) '
         f'|> filter(fn: (r) => r._measurement == "{measurement}") |> last()')
    n = 0
    for line in flux_query(q).splitlines():
        p = line.split(",")
        if len(p) > 7 and p[1] == "_result":
            print(f"  {p[7]:28s} = {p[6]}")
            n += 1
    if n == 0:
        print("  (no data in window — is docker compose up?)")

print("Helpers loaded.")


Helpers loaded.


In [2]:
print("robot_telemetry (drives pose, sensors, battery panels + 3D robot):")
show_last("robot_telemetry")
print()
print("robot_state (drives status strip, coverage gauge + 3D colours):")
show_last("robot_state")


robot_telemetry (drives pose, sensors, battery panels + 3D robot):
  battery_a                    = 1.5
  battery_soc                  = 97.18
  battery_v                    = 12.527
  brush_on                     = 1
  bumper_active                = 0
  dirt_score                   = 0.7224
  heading_deg                  = 90
  motor_current_a              = 0.9
  motor_temperature_c          = 26.5
  obstacle_cm                  = 200
  pump_on                      = 0
  sequence                     = 2416
  speed_mps                    = 0.2
  water_level_pct              = 100
  x_m                          = 3
  y_m                          = 4

robot_state (drives status strip, coverage gauge + 3D colours):
  alarm_count                  = 0
  battery_state                = NORMAL
  cleaning_coverage_pct        = 54.24
  connection_state             = ONLINE
  dirt_level                   = DIRTY
  mission_state                = RUNNING
  motion_state                 = MOVING
  m

## 5. Visualization reacting to the twin — fault demonstration

Injecting a motor fault makes the dashboard status strip turn red/orange and
the Omniverse robot body turn red — both within ~5 seconds, both from the
same state change. (Screenshots of before/during/after in `docs/evidence/`.)


In [3]:
def inject(fault):
    req = urllib.request.Request("http://localhost:8004/fault",
        data=json.dumps({"fault": fault}).encode(),
        headers={"Content-Type": "application/json"}, method="POST")
    with urllib.request.urlopen(req, timeout=5) as r:
        print(f"POST /fault {fault!r} -> HTTP {r.status}")

inject("motor")
time.sleep(15)
print("\nTwin state during fault:")
show_last("robot_state", 10)
inject("clear")
print("\nFault cleared — visualizations return to green within seconds.")


POST /fault 'motor' -> HTTP 200



Twin state during fault:
  alarm_count                  = 1
  battery_state                = NORMAL
  cleaning_coverage_pct        = 61.02
  connection_state             = ONLINE
  dirt_level                   = DIRTY
  mission_state                = RUNNING
  motion_state                 = MOVING
  motor_health                 = HIGH_LOAD
  safety_state                 = SAFE
  twin_quality                 = SYNCHRONIZED
POST /fault 'clear' -> HTTP 200

Fault cleared — visualizations return to green within seconds.


## Rubric Mapping — Skilled (5) Level

| Skilled (5) criterion | Where demonstrated in this submission |
|---|---|
| Clearly & persuasively defines problem statement and purpose (SMART outcome) | Section 1 — problem, SMART purpose |
| Selected sensor data | Section 4 live telemetry (16 measurables) driving every panel |
| Digital twin state displayed | Section 4 — 11-dimension `robot_state` feeding status strip & 3D colours |
| **Measure of success displayed in visualization** | Cleaning coverage % gauge + progress trend (Section 2), tiles in 3D (Section 3) |
| Working demo of Grafana | 28-panel live dashboard, 5 s refresh, provisioned as code; fault reaction demo in Section 5 |
| Data & visualization aligned to twin purpose | Section 2 table — every panel mapped to a purpose |
